# Analyze `data.jsonl` Size Risks

Use this notebook before running Joern-heavy pipelines. It scans each benchmark `data.jsonl` in a streaming-friendly way and reports very large Java methods that may slow down parsing, produce huge graphs, or increase crash risk.

The goal is not to delete anything automatically. The goal is to identify outliers so we can decide whether to keep them, cap them, chunk more aggressively, or inspect them manually.

In [1]:
from pathlib import Path
import heapq
import json
import math
import statistics

import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATASETS = {
    "type1": PROJECT_ROOT / "bench_data" / "bcb_full_type1" / "data.jsonl",
    "type2": PROJECT_ROOT / "bench_data" / "bcb_full_type2" / "data.jsonl",
}

TOP_N = 50

# Tune these thresholds after seeing your machine's behavior.
RISK_CHARS = 50_000
RISK_LINES = 1_000
RISK_LONGEST_LINE = 10_000

DATASETS

{'type1': WindowsPath('c:/Users/koush/PyProjects/Spectral-Software/bench_data/bcb_full_type1/data.jsonl'),
 'type2': WindowsPath('c:/Users/koush/PyProjects/Spectral-Software/bench_data/bcb_full_type2/data.jsonl')}

## Streaming Scanner

This reads one JSONL row at a time. It keeps summary statistics, risk counts, and the top `TOP_N` largest methods by character count.

In [2]:
def percentile(values, q):
    if not values:
        return 0
    values = sorted(values)
    pos = (len(values) - 1) * q
    lo = math.floor(pos)
    hi = math.ceil(pos)
    if lo == hi:
        return values[lo]
    return values[lo] * (hi - pos) + values[hi] * (pos - lo)


def scan_data_jsonl(path: Path, top_n: int = TOP_N):
    if not path.exists():
        raise FileNotFoundError(path)

    char_counts = []
    line_counts = []
    longest_line_counts = []
    top_heap = []
    risky_rows = []
    duplicate_ids = 0
    seen_ids = set()
    malformed_rows = 0

    with path.open("r", encoding="utf-8") as f:
        for row_no, line in enumerate(f, start=1):
            try:
                record = json.loads(line)
            except json.JSONDecodeError:
                malformed_rows += 1
                continue

            idx = str(record.get("idx", ""))
            code = record.get("func", "") or ""
            if idx in seen_ids:
                duplicate_ids += 1
            seen_ids.add(idx)

            n_chars = len(code)
            lines = code.splitlines() or [""]
            n_lines = len(lines)
            longest_line = max(len(x) for x in lines)
            non_ascii = sum(1 for ch in code if ord(ch) > 127)
            brace_count = code.count("{") + code.count("}")

            char_counts.append(n_chars)
            line_counts.append(n_lines)
            longest_line_counts.append(longest_line)

            item = (n_chars, row_no, idx, n_lines, longest_line, non_ascii, brace_count)
            if len(top_heap) < top_n:
                heapq.heappush(top_heap, item)
            elif item > top_heap[0]:
                heapq.heapreplace(top_heap, item)

            reasons = []
            if n_chars >= RISK_CHARS:
                reasons.append("very_many_chars")
            if n_lines >= RISK_LINES:
                reasons.append("very_many_lines")
            if longest_line >= RISK_LONGEST_LINE:
                reasons.append("very_long_line")
            if reasons:
                risky_rows.append({
                    "idx": idx,
                    "row_no": row_no,
                    "chars": n_chars,
                    "lines": n_lines,
                    "longest_line": longest_line,
                    "non_ascii": non_ascii,
                    "brace_count": brace_count,
                    "reasons": ",".join(reasons),
                })

    total_chars = sum(char_counts)
    summary = {
        "path": str(path),
        "rows": len(char_counts),
        "malformed_rows": malformed_rows,
        "duplicate_ids": duplicate_ids,
        "total_chars": total_chars,
        "mean_chars": statistics.mean(char_counts) if char_counts else 0,
        "p50_chars": percentile(char_counts, 0.50),
        "p95_chars": percentile(char_counts, 0.95),
        "p99_chars": percentile(char_counts, 0.99),
        "max_chars": max(char_counts) if char_counts else 0,
        "max_lines": max(line_counts) if line_counts else 0,
        "max_longest_line": max(longest_line_counts) if longest_line_counts else 0,
        "risky_rows": len(risky_rows),
    }

    top_rows = [
        {
            "idx": idx,
            "row_no": row_no,
            "chars": n_chars,
            "lines": n_lines,
            "longest_line": longest_line,
            "non_ascii": non_ascii,
            "brace_count": brace_count,
        }
        for n_chars, row_no, idx, n_lines, longest_line, non_ascii, brace_count in sorted(top_heap, reverse=True)
    ]

    return summary, pd.DataFrame(top_rows), pd.DataFrame(risky_rows)

## Run The Scan

In [3]:
summaries = []
top_by_dataset = {}
risky_by_dataset = {}

for dataset_name, data_path in DATASETS.items():
    print(f"Scanning {dataset_name}: {data_path}")
    summary, top_df, risky_df = scan_data_jsonl(data_path)
    summary["dataset"] = dataset_name
    summaries.append(summary)
    top_by_dataset[dataset_name] = top_df
    risky_by_dataset[dataset_name] = risky_df

summary_df = pd.DataFrame(summaries).set_index("dataset")
summary_df

Scanning type1: c:\Users\koush\PyProjects\Spectral-Software\bench_data\bcb_full_type1\data.jsonl
Scanning type2: c:\Users\koush\PyProjects\Spectral-Software\bench_data\bcb_full_type2\data.jsonl


,path,rows,malformed_rows,duplicate_ids,total_chars,mean_chars,p50_chars,p95_chars,p99_chars,max_chars,max_lines,max_longest_line,risky_rows
dataset,,,,,,,,,,,,,
type1,c:\Users\koush\PyProjects\Spectral-Software\be...,107413,0,0,40382119,375.951877,136.0,1348.0,3554.00,107178,2003,15480,8
type2,c:\Users\koush\PyProjects\Spectral-Software\be...,191732,0,0,66592034,347.318309,131.0,1220.0,3121.69,107178,2003,15480,12


## Largest Methods

These are the methods most likely to create large AST/CFG/DDG/PDG exports. Start here if Joern fails or spectral extraction becomes slow.

In [4]:
for dataset_name, top_df in top_by_dataset.items():
    print(f"\n=== {dataset_name}: largest methods ===")
    display(top_df.head(20))


=== type1: largest methods ===


,idx,row_no,chars,lines,longest_line,non_ascii,brace_count
0,1263802,7205,107178,953,179,0,6
1,8320573,38625,69144,615,1024,27,54
2,11612452,53174,62657,1223,266,0,354
3,11609011,53157,59235,440,1023,0,30
4,12212280,55788,55694,1022,200,9,146
5,19322918,87649,53369,46,15480,0,138
6,15276380,69562,50935,1176,131,3,342
7,18276378,83032,44438,882,263,0,418
8,20363513,92284,41272,560,1023,0,120
9,12733653,57993,38960,402,954,0,128



=== type2: largest methods ===


,idx,row_no,chars,lines,longest_line,non_ascii,brace_count
0,1263802,9249,107178,953,179,0,6
1,8320573,66448,69144,615,1024,27,54
2,11612452,93001,62657,1223,266,0,354
3,11609011,92967,59235,440,1023,0,30
4,386271,2904,56974,869,126,0,104
5,21628276,174406,56850,774,335,0,2
6,12212280,97811,55694,1022,200,9,146
7,19322918,155866,53369,46,15480,0,138
8,15276380,122756,50935,1176,131,3,342
9,18276378,147348,44438,882,263,0,418


## Risk Rows

Rows here crossed at least one threshold. This does not mean they are wrong. It means they deserve attention before a long Joern run.

In [5]:
for dataset_name, risky_df in risky_by_dataset.items():
    print(f"\n=== {dataset_name}: risky rows ===")
    if risky_df.empty:
        print("No rows crossed the current thresholds.")
    else:
        display(risky_df.sort_values(["chars", "lines", "longest_line"], ascending=False).head(50))


=== type1: risky rows ===


,idx,row_no,chars,lines,longest_line,non_ascii,brace_count,reasons
1,1263802,7205,107178,953,179,0,6,very_many_chars
2,8320573,38625,69144,615,1024,27,54,very_many_chars
4,11612452,53174,62657,1223,266,0,354,"very_many_chars,very_many_lines"
3,11609011,53157,59235,440,1023,0,30,very_many_chars
5,12212280,55788,55694,1022,200,9,146,"very_many_chars,very_many_lines"
7,19322918,87649,53369,46,15480,0,138,"very_many_chars,very_long_line"
6,15276380,69562,50935,1176,131,3,342,"very_many_chars,very_many_lines"
0,634878,4138,27043,2003,28,0,2,very_many_lines



=== type2: risky rows ===


,idx,row_no,chars,lines,longest_line,non_ascii,brace_count,reasons
2,1263802,9249,107178,953,179,0,6,very_many_chars
3,8320573,66448,69144,615,1024,27,54,very_many_chars
5,11612452,93001,62657,1223,266,0,354,"very_many_chars,very_many_lines"
4,11609011,92967,59235,440,1023,0,30,very_many_chars
0,386271,2904,56974,869,126,0,104,very_many_chars
11,21628276,174406,56850,774,335,0,2,very_many_chars
7,12212280,97811,55694,1022,200,9,146,"very_many_chars,very_many_lines"
9,19322918,155866,53369,46,15480,0,138,"very_many_chars,very_long_line"
8,15276380,122756,50935,1176,131,3,342,"very_many_chars,very_many_lines"
10,19962035,160954,32634,1080,160,0,358,very_many_lines


## Inspect One Method By ID

Set `DATASET_NAME` and `METHOD_ID`, then run the cell. This prints a bounded preview so the notebook does not become enormous.

In [6]:
DATASET_NAME = "type2"
METHOD_ID = ""  # Example: paste an idx from the tables above.
PREVIEW_CHARS = 6_000

def load_method(path: Path, method_id: str):
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            record = json.loads(line)
            if str(record.get("idx")) == str(method_id):
                return record.get("func", "") or ""
    return None

if METHOD_ID:
    code = load_method(DATASETS[DATASET_NAME], METHOD_ID)
    if code is None:
        print(f"Method {METHOD_ID} not found in {DATASET_NAME}.")
    else:
        print(f"idx={METHOD_ID} chars={len(code):,} lines={len(code.splitlines()):,}")
        print(code[:PREVIEW_CHARS])
        if len(code) > PREVIEW_CHARS:
            print(f"\n... truncated {len(code) - PREVIEW_CHARS:,} chars ...")
else:
    print("Set METHOD_ID first.")

Set METHOD_ID first.


## Optional: Save Reports

Run this only if you want CSV files beside each benchmark. These files are diagnostic artifacts and can be regenerated.

In [ ]:
SAVE_REPORTS = False

if SAVE_REPORTS:
    for dataset_name, data_path in DATASETS.items():
        out_dir = data_path.parent
        top_by_dataset[dataset_name].to_csv(out_dir / "data_jsonl_largest_methods.csv", index=False)
        risky_by_dataset[dataset_name].to_csv(out_dir / "data_jsonl_risky_methods.csv", index=False)
    summary_df.to_csv(PROJECT_ROOT / "bench_data" / "data_jsonl_size_summary.csv")
    print("Reports saved.")
else:
    print("SAVE_REPORTS is False. No files written.")

SAVE_REPORTS is False. No files written.


: 